comparing masking method performances using vgg16

using cifar10 and cifar100 datasets

In [ ]:
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
from torchvision.transforms import v2
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import random
import torch.autograd as autograd
import math
from matplotlib import pyplot as plt
from itertools import combinations_with_replacement
from torch.utils.data import Dataset

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

In [ ]:
import os
import csv
import json
from datetime import datetime

def make_run_dir(base="/content/drive/MyDrive/experiments/vgg_method_comp", run_name=None):
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    if run_name is None:
        run_name = timestamp
    run_dir = os.path.join(base, run_name)
    os.makedirs(run_dir, exist_ok=True)
    return run_dir

def save_losses_csv(losses, filepath):
    with open(filepath, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["step", "loss"])
        for step, loss in enumerate(losses):
            writer.writerow([step, loss])

def save_metrics_json(metrics_dict, filepath):
    with open(filepath, "w") as f:
        json.dump(metrics_dict, f, indent=2)

def save_model(model, filepath):
    torch.save(model.state_dict(), filepath)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# wrap any dataset with this since we don't
# need to do any transform during training
class PreloadedDataset(Dataset):
    def __init__(self, dataset):
        self.images = []
        self.labels = []

        for i in range(len(dataset)):
            # DOES TRANSFORM UP FRONT BEFORE TRAINING
            # this is why the training loop goes faster
            img, label = dataset[i]

            self.images.append(img)
            self.labels.append(label)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

def getTrainingDataLoaders(
    dataset_name,
    download=True,
    BATCH_SIZE=64,
    augment=True,
    preload_train=None,
    half_seed=None
):
    dataset_name = dataset_name.lower()

    # ----------------------------
    # CIFAR branch
    # ----------------------------
    if dataset_name in ["cifar10", "cifar100"]:
        if dataset_name == "cifar10":
            DataClass = datasets.CIFAR10
            n_classes = 10
        else:
            DataClass = datasets.CIFAR100
            n_classes = 100

        task = "multi-class"
        info = {
            "dataset": dataset_name,
            "n_channels": 3,
            "size": (32, 32)
        }

        # normal eval/test transform
        test_transform = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

        # standard random train augmentation for CIFAR
        if augment:
            train_transform = v2.Compose([
                v2.ToImage(),
                v2.RandomHorizontalFlip(p=0.5),
                v2.RandomCrop(size=(32, 32), padding=4),
                # optional:
                # v2.RandomRotation(15),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])
        else:
            train_transform = test_transform

        train_dataset = DataClass(
            root="./data",
            train=True,
            transform=train_transform,
            download=download
        )
        test_dataset = DataClass(
            root="./data",
            train=False,
            transform=test_transform,
            download=download
        )

    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}")


    # automatic choice:
    # if augment=True, don't preload, so random augmentation happens every epoch
    if preload_train is None:
        preload_train = not augment

    if preload_train:
        train_dataset = PreloadedDataset(train_dataset)

    train_loader = data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    train_loader_at_eval = data.DataLoader(train_dataset, batch_size=2 * BATCH_SIZE, shuffle=False)
    test_loader = data.DataLoader(test_dataset, batch_size=2 * BATCH_SIZE, shuffle=False)

    return info, task, n_classes, train_loader, train_loader_at_eval, test_loader

training and testing helpers

In [ ]:
def get_effective_sparsity_info(model):
    total_zeros = 0
    total_count = 0

    for module in model.modules():
        if isinstance(module, PoppedUpLayer) and isinstance(module.module, (nn.Linear, nn.Conv2d)):
            effective_weight = module._masked_parameters()["weight"]
            total_zeros += (effective_weight == 0).sum().item()
            total_count += effective_weight.numel()

    return {"sparsity": total_zeros / total_count if total_count > 0 else 0.0,
            "zero_count":total_count,
            "total_count": total_count}

def trainit(model,
            NUM_EPOCHS,
            train_loader,
            optimizer,
            task,
            n_classes,
            return_losses=False,
            no_progress=False,
            return_sparsity=False):

    if task == "multi-label, binary-class":
        criterion = nn.BCEWithLogitsLoss()
    else:
        criterion = nn.CrossEntropyLoss()

    if return_losses:
        losses = []

    if return_sparsity:
        sparsities = []

    for epoch in range(NUM_EPOCHS):
        print(f"{epoch / NUM_EPOCHS * 100:.1f}% DONE")
        model.train()

        loader = train_loader if no_progress else tqdm(train_loader)

        for inputs, targets in loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(inputs)[:, :n_classes]

            if task == "multi-label, binary-class":
                targets = targets.to(torch.float32)
                loss = criterion(outputs, targets)
            else:
                targets = targets.long().view(-1)
                loss = criterion(outputs, targets)

            loss.backward()
            optimizer.step()

            if return_losses:
                losses.append(loss.item())

            if return_sparsity:
                sparsities.append(get_effective_sparsity_info(model)["sparsity"])

    if return_losses and return_sparsity:
        return losses, sparsities
    elif return_losses:
        return losses
    elif return_sparsity:
        return sparsities

def train_with_epoch_checkpoints(
    model,
    total_epochs,
    checkpoint_epochs_input, # Renamed parameter for clarity
    train_loader,
    optimizer,
    task,
    n_classes,
    train_loader_at_eval,
    test_loader,
    data_flag,
    run_dir,
    prefix="train",
    no_progress=False,
    return_losses=True,
    return_sparsity=True,
):
    """
    Train in chunks so we can evaluate at specific epoch counts.

    This is designed to match the behavior of repeated calls to your
    original trainit(...), while keeping the same optimizer/model state.

    Returns:
        all_losses, all_sparsities, checkpoint_rows
    """

    # Ensure unique and sorted checkpoints, including 0 if present
    checkpoint_epochs_input = sorted(list(set(checkpoint_epochs_input)))

    all_losses = [] if return_losses else None
    all_sparsities = [] if return_sparsity else None
    checkpoint_rows = []

    trained_so_far = 0

    # Handle epoch 0 evaluation separately if requested
    if 0 in checkpoint_epochs_input:
        print("Evaluating model at epoch 0 (pre-training)...")
        metrics = evaluate_at_epoch(
            model=model,
            epoch=0,
            train_loader_at_eval=train_loader_at_eval,
            test_loader=test_loader,
            task=task,
            n_classes=n_classes,
            data_flag=data_flag,
        )
        checkpoint_rows.append(metrics)

        save_metrics_json(
            metrics,
            os.path.join(run_dir, f"{prefix}_metrics_epoch_0.json")
        )

    # Filter for positive checkpoints for the training loop
    positive_checkpoints = sorted(set(e for e in checkpoint_epochs_input if e > 0 and e <= total_epochs))
    if not positive_checkpoints or positive_checkpoints[-1] != total_epochs:
        positive_checkpoints.append(total_epochs)
        positive_checkpoints = sorted(list(set(positive_checkpoints))) # Sort again to maintain order

    for target_epoch in positive_checkpoints:
        chunk_epochs = target_epoch - trained_so_far
        if chunk_epochs <= 0:
            continue

        # call trainit using the same flags/behavior as your original function
        out = trainit(
            model,
            chunk_epochs,
            train_loader,
            optimizer,
            task=task,
            n_classes=n_classes,
            return_losses=return_losses,
            no_progress=no_progress,
            return_sparsity=return_sparsity
        )

        # unpack exactly according to trainit's return style
        if return_losses and return_sparsity:
            chunk_losses, chunk_sparsities = out
            if all_losses is not None: all_losses.extend(chunk_losses)
            if all_sparsities is not None: all_sparsities.extend(chunk_sparsities)
        elif return_losses:
            chunk_losses = out
            if all_losses is not None: all_losses.extend(chunk_losses)
        elif return_sparsity:
            chunk_sparsities = out
            if all_sparsities is not None: all_sparsities.extend(chunk_sparsities)

        trained_so_far = target_epoch

        metrics = evaluate_at_epoch(
            model=model,
            epoch=trained_so_far,
            train_loader_at_eval=train_loader_at_eval,
            test_loader=test_loader,
            task=task,
            n_classes=n_classes,
            data_flag=data_flag
        )
        checkpoint_rows.append(metrics)

        save_metrics_json(
            metrics,
            os.path.join(run_dir, f"{prefix}_metrics_epoch_{trained_so_far}.json")
        )

    save_checkpoint_metrics_csv(
        checkpoint_rows,
        os.path.join(run_dir, f"{prefix}_checkpoint_metrics.csv")
    )

    return all_losses, all_sparsities, checkpoint_rows

def test(split,
         model,
         train_loader_at_eval,
         test_loader,
         task,
         n_classes,
         data_flag=None,
         return_metrics=False):

    model.eval()
    criterion = nn.CrossEntropyLoss()

    data_loader = train_loader_at_eval if split == 'train' else test_loader

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True).long()   # shape [B]

            outputs = model(inputs)[:, :n_classes]   # shape [B, n_classes]
            loss = criterion(outputs, targets)

            preds = outputs.argmax(dim=1)

            batch_size = targets.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (preds == targets).sum().item()
            total_samples += batch_size

    avg_loss = total_loss / total_samples
    acc = total_correct / total_samples

    print(f"{split} loss: {avg_loss:.4f} acc: {acc:.4f}")

    if return_metrics:
        return avg_loss, acc

In [ ]:
import torch.nn as nn
from torch import autograd
from torch.func import functional_call

class ClassicNetwork(nn.Module):
    def __init__(self, layer_sizes,bias=True):
        super().__init__()

        self.flatten = nn.Flatten()

        self.layers = nn.Sequential(
            *[z for l in layer_sizes
              for z in [nn.Linear(l[0], l[1],bias=bias), nn.ReLU()]][:-1]
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.layers(x)
        return logits


class GetSubnet(autograd.Function):

    @staticmethod
    def forward(ctx, scores, k):

        # Get the subnetwork by sorting the scores and using the top k%
        out = scores.clone()
        _, idx = scores.flatten().sort()
        j = int((1-k) * scores.numel())

        # flat_out and out access the same memory.
        flat_out = out.flatten()
        flat_out[idx[:j]] = 0
        flat_out[idx[j:]] = 1

        return out

    @staticmethod
    def backward(ctx, grad):

        # send the gradient g straight-through on the backward pass.
        return grad, None

"""
wrapper to turn module into module using popup
ie freeze the params and give doubled popups
change forward pass, etc
"""
class PoppedUpLayer(nn.Module):
    def __init__(self, module: nn.Module, k: float = 0.5, just_weight: bool = True):
        super().__init__()
        self.module = module
        self.k = k
        self.just_weight = just_weight

        # Freeze original module params by default
        for p in self.module.parameters():
            p.requires_grad_(False)

        self.popup_scores = nn.ParameterDict()

        for name, param in module.named_parameters():
            # If just_weight=True, only create popup scores for the weight param
            if self.just_weight and name != "weight":
                continue

            score = nn.Parameter(torch.randn(2 * param.shape[0], *param.shape[1:])).to(param.device)
            safe_name = name.replace(".", "__")
            self.popup_scores[safe_name] = score

    def _score_key(self, name: str) -> str:
        return name.replace(".", "__")

    # ----- using this stuff for training after pruned ---
    def freeze_popup_scores(self):
        for p in self.popup_scores.parameters():
            p.requires_grad_(False)

    def unfreeze_popup_scores(self):
        for p in self.popup_scores.parameters():
            p.requires_grad_(True)

    def freeze_module_params(self):
        for p in self.module.parameters():
            p.requires_grad_(False)

    def unfreeze_module_params(self):
        for p in self.module.parameters():
            p.requires_grad_(True)

    def set_mask_training_mode(self):
        self.unfreeze_popup_scores()
        self.freeze_module_params()

    def set_subnetwork_training_mode(self):
        self.freeze_popup_scores()
        self.unfreeze_module_params()
    # -----------------------------------------------

    def _masked_parameters(self):
        masked_params = {}

        for name, param in self.module.named_parameters():
            # Only mask weight if just_weight=True
            if self.just_weight and name != "weight":
                masked_params[name] = param
                continue

            score = self.popup_scores[self._score_key(name)]
            mask = GetSubnet.apply(score.abs(), self.k)[:param.shape[0]]
            masked_params[name] = param * mask

        return masked_params

    def forward(self, x):
        masked_params = self._masked_parameters()
        buffers = dict(self.module.named_buffers())
        return functional_call(self.module, {**masked_params, **buffers}, (x,))


class MaskLayer(nn.Module):
    def __init__(self, layer: nn.Module, mask: torch.Tensor | None = None):
        super().__init__()

        if not isinstance(layer, (nn.Linear, nn.Conv2d)):
            raise TypeError("MaskLayer only supports nn.Linear and nn.Conv2d")

        self.layer = layer

        if mask is None:
            mask = torch.ones_like(layer.weight)
        else:
            mask = mask.detach().clone().to(layer.weight.device, dtype=layer.weight.dtype)

        if mask.shape != layer.weight.shape:
            raise ValueError(
                f"Mask shape {mask.shape} must match weight shape {layer.weight.shape}"
            )

        self.register_buffer("mask", mask)

    @property
    def weight(self):
        return self.layer.weight

    @property
    def bias(self):
        return self.layer.bias

    def forward(self, x):
        masked_weight = self.layer.weight * self.mask
        bias = self.layer.bias

        if isinstance(self.layer, nn.Linear):
            return F.linear(x, masked_weight, bias)

        elif isinstance(self.layer, nn.Conv2d):
            return F.conv2d(
                x,
                masked_weight,
                bias,
                stride=self.layer.stride,
                padding=self.layer.padding,
                dilation=self.layer.dilation,
                groups=self.layer.groups,
            )

        raise RuntimeError("Unsupported layer type in MaskLayer")


class MaskedNetwork(nn.Module):
    def __init__(self, net: nn.Module, masks=None):
        super().__init__()

        self.net = copy.deepcopy(net)

        if masks is None:
            masks = []

        self._mask_index = 0
        self._provided_masks = masks

        self._wrap_modules(self.net)


    def _next_mask(self):
        if self._mask_index < len(self._provided_masks):
            mask = self._provided_masks[self._mask_index]
        else:
            mask = None
        self._mask_index += 1
        return mask

    def _wrap_modules(self, module: nn.Module):
        for name, child in list(module.named_children()):
            if isinstance(child, (nn.Linear, nn.Conv2d)):
                mask = self._next_mask()
                setattr(module, name, MaskLayer(child, mask))
            else:
                self._wrap_modules(child)

    def forward(self, x):
        return self.net(x)

    def get_masks(self):
        masks = []
        for m in self.net.modules():
            if isinstance(m, MaskLayer):
                masks.append(m.mask)
        return masks






In [ ]:
import copy
import torch.nn as nn

def is_popupifiable(module: nn.Module) -> bool:
    # only want to popupify these, ie ignore batchnorm
    return isinstance(module, (nn.Linear, nn.Conv2d))

def popupify_inplace(module: nn.Module, k=.5):
    for name, child in list(module.named_children()):
        # recurse first so we reach leaves inside bigger blocks
        popupify_inplace(child, k)

        # only replace Conv2d and Linear layers
        if is_popupifiable(child):
            setattr(module, name, PoppedUpLayer(child, k))

    return module

def popupify(network: nn.Module, k=.5):
    net_copy = copy.deepcopy(network)
    return popupify_inplace(net_copy, k)

def set_subnetwork_training_mode(model):
    for m in model.modules():
        if isinstance(m, PoppedUpLayer):
            m.set_subnetwork_training_mode()

In [ ]:
import torch
import torch.nn as nn
from torch.optim import SGD
from torch.optim.lr_scheduler import MultiStepLR


class CIFARVGG16(nn.Module):
    """
    CIFAR-style VGG16 matching the https://arxiv.org/pdf/2009.11094 setup:
    - conv bias=False
    - BatchNorm + ReLU after each conv
    - no final max-pool after the last 512 block
    - AvgPool2d(2) before the classifier
    - single linear classifier: 512 -> num_classes
    """

    cfg = [
        64, 64, "M",
        128, 128, "M",
        256, 256, 256, "M",
        512, 512, 512, "M",
        512, 512, 512
    ]

    def __init__(self, num_classes: int = 10, batchnorm: bool = True, affine: bool = True):
        super().__init__()
        self.features = self._make_layers(self.cfg, batchnorm=batchnorm, affine=affine)
        self.avgpool = nn.AvgPool2d(kernel_size=2)
        self.classifier = nn.Linear(512, num_classes)
        self._initialize_weights()

    def _make_layers(self, cfg, batchnorm: bool, affine: bool):
        layers = []
        in_channels = 3

        for v in cfg:
            if v == "M":
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                conv = nn.Conv2d(
                    in_channels=in_channels,
                    out_channels=v,
                    kernel_size=3,
                    padding=1,
                    bias=False
                )
                if batchnorm:
                    layers.extend([
                        conv,
                        nn.BatchNorm2d(v, affine=affine),
                        nn.ReLU(inplace=True),
                    ])
                else:
                    layers.extend([
                        conv,
                        nn.ReLU(inplace=True),
                    ])
                in_channels = v

        return nn.Sequential(*layers)

    def _initialize_weights(self):
        # Kaiming-style conv init; BN scale=1, bias=0; linear small normal init
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                if m.weight is not None:
                    nn.init.constant_(m.weight, 1.0)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.01)
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = self.features(x)      # [B, 512, 2, 2] for CIFAR-10 32x32
        x = self.avgpool(x)       # [B, 512, 1, 1]
        x = torch.flatten(x, 1)   # [B, 512]
        x = self.classifier(x)    # [B, 10]
        return x



In [ ]:

import torch
import torch.nn as nn

class CIFARMLP(nn.Module):
    def __init__(self, hidden_sizes=[512, 256], num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()

        layers = []
        in_dim = 3 * 32 * 32   # CIFAR input size = 3072

        for h in hidden_sizes:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            in_dim = h

        layers.append(nn.Linear(in_dim, num_classes))
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        # allow single image [3,32,32] or batch [B,3,32,32]
        if x.dim() == 3:
            x = x.unsqueeze(0)

        x = self.flatten(x)
        x = self.layers(x)
        return x

SNIP IMPLEMENTATION

In [ ]:
#https://github.com/JingtongSu/sanity-checking-pruning/blob/main/pruner/SNIP.py
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import math

import copy
import types

def SNIP_fetch_data(dataloader, num_classes, samples_per_class):
    datas = [[] for _ in range(num_classes)]
    labels = [[] for _ in range(num_classes)]
    mark = dict()
    dataloader_iter = iter(dataloader)
    while True:
        inputs, targets = next(dataloader_iter)
        for idx in range(inputs.shape[0]):
            x, y = inputs[idx:idx+1], targets[idx:idx+1]
            category = y.item()
            if len(datas[category]) == samples_per_class:
                mark[category] = True
                continue
            datas[category].append(x)
            labels[category].append(y)
        if len(mark) == num_classes:
            break

    X, y = torch.cat([torch.cat(_, 0) for _ in datas]), torch.cat([torch.cat(_) for _ in labels]).view(-1)
    return X, y


def count_total_parameters(net):
    total = 0
    for m in net.modules():
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            total += m.weight.numel()
    return total


def count_fc_parameters(net):
    total = 0
    for m in net.modules():
        if isinstance(m, (nn.Linear)):
            total += m.weight.numel()
    return total


def SNIP(net, ratio, train_dataloader, device, num_classes=10, samples_per_class=25,num_iters=1):
    eps = 1e-10
    keep_ratio = 1-ratio
    old_net = net

    net = copy.deepcopy(net)  # .eval()
    net.zero_grad()

    weights = []
    total_parameters = count_total_parameters(net)
    fc_parameters = count_fc_parameters(net)

    # rescale_weights(net)
    for layer in net.modules():
        if isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear):
            weights.append(layer.weight)

    inputs_one = []
    targets_one = []

    grad_w = None
    for w in weights:
        w.requires_grad_(True)

    print_once = False

    # num_iters pretraining
    for it in range(num_iters):
        inputs, targets = SNIP_fetch_data(train_dataloader, num_classes, samples_per_class)
        N = inputs.shape[0]
        din = copy.deepcopy(inputs)
        dtarget = copy.deepcopy(targets)
        inputs = inputs.to(device)
        targets = targets.to(device)
        outputs = net.forward(inputs)
        loss = F.cross_entropy(outputs,targets)
        loss.backward()

    # saliency scores
    grads = dict()
    old_modules = list(old_net.modules())
    for idx, layer in enumerate(net.modules()):
        if isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear):
            grads[old_modules[idx]] = abs(layer.weight.data * layer.weight.grad) # -theta_q g

    # Gather all scores in a single vector and normalise
    all_scores = torch.cat([torch.flatten(x) for x in grads.values()])
    norm_factor = torch.abs(torch.sum(all_scores)) + eps
    print("** norm factor:", norm_factor)
    all_scores.div_(norm_factor)

    num_params_to_kp = int(len(all_scores) * (keep_ratio))
    threshold, _ = torch.topk(all_scores, num_params_to_kp, sorted=True)
    # import pdb; pdb.set_trace()
    acceptable_score = threshold[-1]
    print('** accept: ', acceptable_score)
    keep_masks = dict()
    for m, g in grads.items():
        keep_masks[m] = ((g / norm_factor) >= acceptable_score).float()

    print(torch.sum(torch.cat([torch.flatten(x == 1) for x in keep_masks.values()])))

    return keep_masks

GRASP IMPLEMENTATION

In [ ]:
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import math

import copy
import types

#https://github.com/alecwangcq/GraSP/blob/master/pruner/GraSP.py
def GraSP_fetch_data(dataloader, num_classes, samples_per_class):
    datas = [[] for _ in range(num_classes)]
    labels = [[] for _ in range(num_classes)]
    mark = dict()
    dataloader_iter = iter(dataloader)
    while True:
        inputs, targets = next(dataloader_iter)
        for idx in range(inputs.shape[0]):
            x, y = inputs[idx:idx+1], targets[idx:idx+1]
            category = y.item()
            if len(datas[category]) == samples_per_class:
                mark[category] = True
                continue
            datas[category].append(x)
            labels[category].append(y)
        if len(mark) == num_classes:
            break

    X, y = torch.cat([torch.cat(_, 0) for _ in datas]), torch.cat([torch.cat(_) for _ in labels]).view(-1)
    return X, y


def count_total_parameters(net):
    total = 0
    for m in net.modules():
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            total += m.weight.numel()
    return total


def count_fc_parameters(net):
    total = 0
    for m in net.modules():
        if isinstance(m, (nn.Linear)):
            total += m.weight.numel()
    return total


def GraSP(net, ratio, train_dataloader, device, num_classes=10, samples_per_class=25, num_iters=1, T=200, reinit=False):
    eps = 1e-10
    keep_ratio = 1-ratio
    old_net = net

    net = copy.deepcopy(net)  # .eval()
    net.zero_grad()

    weights = []
    total_parameters = count_total_parameters(net)
    fc_parameters = count_fc_parameters(net)

    # rescale_weights(net)
    for layer in net.modules():
        if isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear):
            if isinstance(layer, nn.Linear) and reinit:
                nn.init.xavier_normal(layer.weight)
            weights.append(layer.weight)

    inputs_one = []
    targets_one = []

    grad_w = None
    for w in weights:
        w.requires_grad_(True)

    print_once = False
    for it in range(num_iters):
        print("(1): Iterations %d/%d." % (it, num_iters))
        inputs, targets = GraSP_fetch_data(train_dataloader, num_classes, samples_per_class)
        N = inputs.shape[0]
        din = copy.deepcopy(inputs)
        dtarget = copy.deepcopy(targets)
        inputs_one.append(din[:N//2])
        targets_one.append(dtarget[:N//2])
        inputs_one.append(din[N // 2:])
        targets_one.append(dtarget[N // 2:])
        inputs = inputs.to(device)
        targets = targets.to(device)

        outputs = net.forward(inputs[:N//2])/T
        if print_once:
            # import pdb; pdb.set_trace()
            x = F.softmax(outputs)
            print(x)
            print(x.max(), x.min())
            print_once = False
        loss = F.cross_entropy(outputs, targets[:N//2])
        # ===== debug ================
        grad_w_p = autograd.grad(loss, weights)
        if grad_w is None:
            grad_w = list(grad_w_p)
        else:
            for idx in range(len(grad_w)):
                grad_w[idx] += grad_w_p[idx]

        outputs = net.forward(inputs[N // 2:])/T
        loss = F.cross_entropy(outputs, targets[N // 2:])
        grad_w_p = autograd.grad(loss, weights, create_graph=False)
        if grad_w is None:
            grad_w = list(grad_w_p)
        else:
            for idx in range(len(grad_w)):
                grad_w[idx] += grad_w_p[idx]

    ret_inputs = []
    ret_targets = []

    for it in range(len(inputs_one)):
        print("(2): Iterations %d/%d." % (it, num_iters))
        inputs = inputs_one.pop(0).to(device)
        targets = targets_one.pop(0).to(device)
        ret_inputs.append(inputs)
        ret_targets.append(targets)
        outputs = net.forward(inputs)/T
        loss = F.cross_entropy(outputs, targets)
        # ===== debug ==============

        grad_f = autograd.grad(loss, weights, create_graph=True)
        z = 0
        count = 0
        for layer in net.modules():
            if isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear):
                z += (grad_w[count].data * grad_f[count]).sum()
                count += 1
        z.backward()

    grads = dict()
    old_modules = list(old_net.modules())
    for idx, layer in enumerate(net.modules()):
        if isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear):
            grads[old_modules[idx]] = -layer.weight.data * layer.weight.grad  # -theta_q Hg

    # Gather all scores in a single vector and normalise
    all_scores = torch.cat([torch.flatten(x) for x in grads.values()])
    norm_factor = torch.abs(torch.sum(all_scores)) + eps
    print("** norm factor:", norm_factor)
    all_scores.div_(norm_factor)

    num_params_to_rm = int(len(all_scores) * (1-keep_ratio))
    threshold, _ = torch.topk(all_scores, num_params_to_rm, sorted=True)
    # import pdb; pdb.set_trace()
    acceptable_score = threshold[-1]
    print('** accept: ', acceptable_score)
    keep_masks = dict()
    for m, g in grads.items():
        keep_masks[m] = ((g / norm_factor) <= acceptable_score).float()

    print(torch.sum(torch.cat([torch.flatten(x == 1) for x in keep_masks.values()])))

    return keep_masks

IMP

In [ ]:
from itertools import cycle

# need new training function to give control over number of steps
def train_step(model, batch, optimizer, task, n_classes, device):
    if task == "multi-label, binary-class":
        criterion = nn.BCEWithLogitsLoss()
    else:
        criterion = nn.CrossEntropyLoss()

    model.train()

    inputs, targets = batch
    inputs = inputs.to(device, non_blocking=True)
    targets = targets.to(device, non_blocking=True)

    optimizer.zero_grad()
    outputs = model(inputs)[:, :n_classes]

    if task == "multi-label, binary-class":
        targets = targets.to(torch.float32)
        loss = criterion(outputs, targets)
    else:
        targets = targets.long().view(-1)
        loss = criterion(outputs, targets)

    loss.backward()
    optimizer.step()

    return loss.item()

def train_for_steps(model,
                    num_steps,
                    train_loader,
                    optimizer,
                    task,
                    n_classes,
                    device,
                    return_losses=False,
                    no_progress=False):

    if return_losses:
        losses = []

    loader = cycle(train_loader)

    iterator = range(num_steps)
    if not no_progress:
        iterator = tqdm(iterator)

    for _ in iterator:
        batch = next(loader)
        loss_val = train_step(
            model=model,
            batch=batch,
            optimizer=optimizer,
            task=task,
            n_classes=n_classes,
            device=device
        )

        if return_losses:
            losses.append(loss_val)

    if return_losses:
        return losses

# helper to get weight sparsity of MaskedNetwork
def maskednetwork_sparsity(masked_net):
    zero_count = 0
    total_count = 0

    for m in masked_net.modules():
        if isinstance(m, MaskLayer):
            mask = m.mask
            zero_count += (mask == 0).sum().item()
            total_count += mask.numel()

    if total_count == 0:
        raise ValueError("No MaskLayer modules found in masked_net")

    sparsity = zero_count / total_count
    return {
        "sparsity": sparsity,
        "zero_count": zero_count,
        "total_count": total_count,
    }



def get_mask_layers(model):
    return [m for m in model.modules() if isinstance(m, MaskLayer)]


def IMP(
    net,
    final_sparsity,
    train_dataloader,
    device,
    tau,
    L_max,
    iter_epochs,
    task,
    n_classes=10,
    train_fn=None,
    rewind_to_init=False,
    prune_global=False,
):
    """
    Iterative Magnitude Pruning:
    train -> prune -> rewind -> repeat
    """

    keep_ratio_final = 1.0 - final_sparsity
    if not (0.0 < keep_ratio_final <= 1.0):
        raise ValueError("final_sparsity must be in [0,1).")

    if L_max < 1:
        raise ValueError("L_max must be >= 1.")

    old_net = net
    base_net = copy.deepcopy(net).to(device)

    # Save initial weights before any training
    base_prunable_layers = [
        m for m in base_net.modules() if isinstance(m, (nn.Linear, nn.Conv2d))
    ]
    if len(base_prunable_layers) == 0:
        raise ValueError("No prunable nn.Linear or nn.Conv2d layers found.")

    init_weights = [layer.weight.detach().clone() for layer in base_prunable_layers]

    # Phase 1: optional rewind point training on unmasked network
    optimizer = optim.Adam(base_net.parameters())

    train_for_steps(
        base_net,
        tau,
        train_dataloader,
        optimizer,
        task,
        n_classes,
        device,
        return_losses=False,
        no_progress=False,
    )

    rewind_weights = [layer.weight.detach().clone() for layer in base_prunable_layers]

    # Wrap AFTER rewind point is obtained
    net = MaskedNetwork(base_net).to(device)
    mask_layers = get_mask_layers(net)

    # Ratio kept per round so that after L_max rounds you hit final sparsity
    round_keep_ratio = keep_ratio_final ** (1.0 / L_max)

    for level in range(L_max):
        print(f"IMP round {level + 1}/{L_max}")

        optimizer = optim.Adam(net.parameters())

        trainit(
            net,
            iter_epochs,
            train_dataloader,
            optimizer,
            task,
            n_classes,
            return_losses=False,
            no_progress=False,
        )

        if prune_global:
            # Global pruning over all currently active weights
            all_masked_mags = []
            all_refs = []

            for layer in mask_layers:
                mask = layer.mask
                weight = layer.weight.detach()
                masked_mags = weight.abs() * mask

                active_idx = (mask > 0).view(-1)
                active_vals = masked_mags.view(-1)[active_idx]

                all_masked_mags.append(active_vals)
                all_refs.append((layer, active_idx))

            all_scores = torch.cat(all_masked_mags)

            total_on = all_scores.numel()
            to_keep_total = int(round_keep_ratio * total_on)
            to_keep_total = max(to_keep_total, 0)

            if to_keep_total == 0:
                threshold = torch.inf
            else:
                sorted_vals, _ = torch.sort(all_scores)
                threshold = sorted_vals[-to_keep_total]

            for layer in mask_layers:
                mask = layer.mask
                masked_mags = layer.weight.detach().abs() * mask
                new_mask = ((masked_mags >= threshold) & (mask > 0)).to(mask.dtype)
                layer.mask.copy_(new_mask)

        else:
            # Layerwise pruning
            for layer in mask_layers:
                mask = layer.mask
                weight = layer.weight.detach()

                masked_mags = weight.abs() * mask
                flat_mask = mask.view(-1)
                flat_mags = masked_mags.view(-1)

                on_idx = torch.nonzero(flat_mask > 0, as_tuple=False).flatten()
                n_on = on_idx.numel()

                if n_on == 0:
                    continue

                to_keep = int(round_keep_ratio * n_on)
                to_keep = max(to_keep, 0)

                if to_keep == 0:
                    flat_mask[on_idx] = 0
                    continue

                active_mags = flat_mags[on_idx]
                _, order = torch.sort(active_mags)

                prune_idx = on_idx[order[:-to_keep]] if to_keep < n_on else torch.tensor([], device=on_idx.device, dtype=on_idx.dtype)
                flat_mask[prune_idx] = 0

        print(f"SPARSITY AFTER PRUNING LVL {level + 1}: {maskednetwork_sparsity(net)}")

        # Rewind surviving weights
        rewind_source = init_weights if rewind_to_init else rewind_weights
        for layer, rewind_w in zip(mask_layers, rewind_source):
            layer.weight.data.copy_(rewind_w.to(layer.weight.device) * layer.mask)

    # Build keep_masks keyed by original model modules
    keep_masks = {}
    old_modules = list(old_net.modules())

    raw_old_prunable = [m for m in old_modules if isinstance(m, (nn.Linear, nn.Conv2d))]

    for old_m, masked_layer in zip(raw_old_prunable, mask_layers):
        keep_masks[old_m] = masked_layer.mask.detach().clone()

    return keep_masks, net


TESTING FUNCTIONS

In [ ]:
import os
import csv
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd

def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
# =========================================================
# small filesystem helpers
# =========================================================
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def save_json(obj, path):
    ensure_dir(Path(path).parent)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def save_list_csv(values, path, column_name="value"):
    ensure_dir(Path(path).parent)
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["step", column_name])
        for i, v in enumerate(values):
            writer.writerow([i, float(v)])


def save_checkpoint_metrics_csv(rows, filepath):
    ensure_dir(Path(filepath).parent)
    with open(filepath, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch",
            "train_loss",
            "train_acc",
            "test_loss",
            "test_acc",
        ])
        for row in rows:
            writer.writerow([
                row["epoch"],
                row["train_loss"],
                row["train_acc"],
                row["test_loss"],
                row["test_acc"],
            ])


def plot_curve(values, title, xlabel, ylabel, label, save_path=None, show=False):
    plt.figure(figsize=(8, 5))
    plt.plot(values, label=label)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()

    if save_path is not None:
        ensure_dir(Path(save_path).parent)
        plt.savefig(save_path, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close()


# =========================================================
# naming helpers
# =========================================================
def make_pruning_run_dir(base_dir, dataset_name, model_type, method_name, seed, final_spar):
    run_dir = os.path.join(
        base_dir,
        f"dataset_{dataset_name}",
        f"model_{model_type}",
        f"method_{method_name}",
        f"sparsity_{final_spar}",
        f"seed_{seed}",
    )
    ensure_dir(run_dir)
    return run_dir


# =========================================================
# evaluation helpers
# =========================================================
def evaluate_model(model, train_loader_at_eval, test_loader, task, n_classes, data_flag):
    train_loss, train_acc = test(
        split="train",
        model=model,
        train_loader_at_eval=train_loader_at_eval,
        test_loader=test_loader,
        task=task,
        n_classes=n_classes,
        data_flag=data_flag,
        return_metrics=True,
    )

    test_loss, test_acc = test(
        split="test",
        model=model,
        train_loader_at_eval=train_loader_at_eval,
        test_loader=test_loader,
        task=task,
        n_classes=n_classes,
        data_flag=data_flag,
        return_metrics=True,
    )

    return {
        "train_loss": float(train_loss),
        "train_acc": float(train_acc),
        "test_loss": float(test_loss),
        "test_acc": float(test_acc),
    }


def evaluate_at_epoch(model, epoch, train_loader_at_eval, test_loader, task, n_classes, data_flag):
    metrics = evaluate_model(
        model=model,
        train_loader_at_eval=train_loader_at_eval,
        test_loader=test_loader,
        task=task,
        n_classes=n_classes,
        data_flag=data_flag,
    )
    metrics["epoch"] = epoch
    return metrics


# =========================================================
# model builder
# =========================================================
def build_model(model_type, n_classes):
    if model_type == "vgg":
        return CIFARVGG16(num_classes=n_classes)
    elif model_type == "mlp":
        return CIFARMLP(num_classes=n_classes)
    elif model_type == "classic_mlp":
        layer_sizes = [[3 * 32 * 32, 256], [256, 256], [256, n_classes]]
        return ClassicNetwork(layer_sizes=layer_sizes, bias=False)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")


# =========================================================
# mask helpers for normal pruning methods
# =========================================================
def ordered_masks_from_keep_masks(model, keep_masks_dict):
    masks = []
    for m in model.modules():
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            if m not in keep_masks_dict:
                raise KeyError("A prunable layer was missing from keep_masks_dict.")
            masks.append(keep_masks_dict[m].detach().clone())
    return masks


def make_dense_keep_masks(model):
    keep_masks = {}
    for m in model.modules():
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            keep_masks[m] = torch.ones_like(m.weight)
    return keep_masks


# =========================================================
# sparsity helpers
# =========================================================
def get_pruned_model_sparsity_info(model):
    """
    For normal MaskedNetwork, use your existing maskednetwork_sparsity.
    For popup model, use get_effective_weight_sparsity.
    For plain dense model, estimate zero fraction in conv/linear weights.
    """
    if isinstance(model, MaskedNetwork):
        return maskednetwork_sparsity(model)

    # popup model branch
    if any(isinstance(m, PoppedUpLayer) for m in model.modules()):

        return get_effective_sparsity_info(model)
# =========================================================
# normal pruning-method dispatcher
# =========================================================
def compute_keep_masks(
    method_name,
    final_spar,
    model,
    train_loader,
    device,
    n_classes,
    task,
    method_kwargs=None,
):
    if method_kwargs is None:
        method_kwargs = {}

    method_name = method_name.lower()



    if method_name == "snip":
        keep_masks = SNIP(
            net=model,
            ratio=final_spar,
            train_dataloader=train_loader,
            device=device,
            num_classes=n_classes,
            samples_per_class=method_kwargs.get("samples_per_class", 25),
            num_iters=method_kwargs.get("num_iters", 1),
        )
        return keep_masks, None

    elif method_name == "grasp":
        keep_masks = GraSP(
            net=model,
            ratio=final_spar,
            train_dataloader=train_loader,
            device=device,
            num_classes=n_classes,
            samples_per_class=method_kwargs.get("samples_per_class", 25),
            num_iters=method_kwargs.get("num_iters", 1),
            T=method_kwargs.get("T", 200),
            reinit=method_kwargs.get("reinit", False),
        )
        return keep_masks, None

    elif method_name == "imp":
        keep_masks, imp_net = IMP(
            net=model,
            final_sparsity=final_spar,
            train_dataloader=train_loader,
            device=device,
            tau=method_kwargs.get("tau", 0),
            L_max=method_kwargs.get("L_max", 10),
            iter_epochs=method_kwargs.get("iter_epochs", 2),
            task=task,
            n_classes=n_classes,
            train_fn=method_kwargs.get("train_fn", None),
            rewind_to_init=method_kwargs.get("rewind_to_init", False),
            prune_global=method_kwargs.get("prune_global", False),
        )
        return keep_masks, imp_net

    else:
        raise ValueError(f"Unknown pruning method: {method_name}")


# =========================================================
# popup pruning phase
# =========================================================
def run_popup_pruning_phase(
    dense_model,
    final_spar,
    train_loader,
    task,
    n_classes,
    run_dir,
    method_kwargs=None,
    no_progress=False,
):
    """
    Popup flow:
      1) popupify dense model
      2) train popup scores with normal trainit(...)
      3) switch to subnetwork training mode
    """
    if method_kwargs is None:
        method_kwargs = {}

    # final_spar = sparsity target, so keep ratio defaults to 1 - sparsity
    popup_k = method_kwargs.get("k", 1.0 - final_spar)
    prune_epochs = method_kwargs.get("prune_epochs", 60)

    popup_model = popupify(dense_model, k=popup_k).to(device)

    prune_optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, popup_model.parameters()),
    )

    prune_losses, prune_sparsities = trainit(
        popup_model,
        prune_epochs,
        train_loader,
        prune_optimizer,
        task=task,
        n_classes=n_classes,
        return_losses=True,
        no_progress=no_progress,
        return_sparsity=True,
    )

    save_list_csv(
        prune_losses,
        os.path.join(run_dir, "popup_prune_losses.csv"),
        column_name="loss",
    )
    save_list_csv(
        prune_sparsities,
        os.path.join(run_dir, "popup_prune_sparsities.csv"),
        column_name="sparsity",
    )

    plot_curve(
        prune_losses,
        title="Popup pruning loss",
        xlabel="Training step",
        ylabel="Loss",
        label="popup prune loss",
        save_path=os.path.join(run_dir, "popup_prune_loss.png"),
        show=False,
    )
    plot_curve(
        prune_sparsities,
        title="Popup effective sparsity during pruning",
        xlabel="Training step",
        ylabel="Sparsity",
        label="popup prune sparsity",
        save_path=os.path.join(run_dir, "popup_prune_sparsity.png"),
        show=False,
    )

    prune_phase_summary = {
        "popup_k": popup_k,
        "prune_epochs": prune_epochs,
        "final_prune_loss_curve_value": float(prune_losses[-1]) if len(prune_losses) else None,
        "final_prune_sparsity_curve_value": float(prune_sparsities[-1]) if len(prune_sparsities) else None,
    }
    save_json(prune_phase_summary, os.path.join(run_dir, "popup_prune_phase_summary.json"))

    # switch from mask training to subnetwork training (train weights now)
    set_subnetwork_training_mode(popup_model)

    return popup_model, prune_losses, prune_sparsities


# =========================================================
# post-pruning training
# =========================================================
def train_pruned_model_with_checkpoints(
    model,
    total_epochs,
    train_loader,
    train_loader_at_eval,
    test_loader,
    optimizer,
    task,
    n_classes,
    data_flag,
    checkpoint_every=5,
    run_dir=None,
    prefix="post_prune",
    no_progress=False,
    return_losses=True,
    return_sparsity=True,
):
    checkpoint_epochs = list(range(0, total_epochs + 1, checkpoint_every))
    if len(checkpoint_epochs) == 0 or checkpoint_epochs[-1] != total_epochs:
        checkpoint_epochs.append(total_epochs)

    train_losses, train_sparsities, checkpoint_rows = train_with_epoch_checkpoints(
        model=model,
        total_epochs=total_epochs,
        checkpoint_epochs_input=checkpoint_epochs,
        train_loader=train_loader,
        optimizer=optimizer,
        task=task,
        n_classes=n_classes,
        train_loader_at_eval=train_loader_at_eval,
        test_loader=test_loader,
        data_flag=data_flag,
        run_dir=run_dir,
        prefix=prefix,
        no_progress=no_progress,
        return_losses=return_losses,
        return_sparsity=return_sparsity,
    )

    if return_losses and train_losses is not None:
        save_list_csv(
            train_losses,
            os.path.join(run_dir, f"{prefix}_train_losses.csv"),
            column_name="loss",
        )
        plot_curve(
            train_losses,
            title=f"{prefix} training loss on {data_flag}",
            xlabel="Training step",
            ylabel="Loss",
            label=f"{prefix} train loss",
            save_path=os.path.join(run_dir, f"{prefix}_train_loss.png"),
            show=False,
        )

    if return_sparsity and train_sparsities is not None:
        save_list_csv(
            train_sparsities,
            os.path.join(run_dir, f"{prefix}_train_sparsities.csv"),
            column_name="sparsity",
        )
        plot_curve(
            train_sparsities,
            title=f"{prefix} sparsity on {data_flag}",
            xlabel="Training step",
            ylabel="Sparsity",
            label=f"{prefix} sparsity",
            save_path=os.path.join(run_dir, f"{prefix}_train_sparsity.png"),
            show=False,
        )

    if checkpoint_rows is not None and len(checkpoint_rows) > 0:
        plot_curve(
            [row["test_acc"] for row in checkpoint_rows],
            title=f"{prefix} test accuracy every {checkpoint_every} epochs",
            xlabel="Checkpoint index",
            ylabel="Accuracy",
            label="test accuracy",
            save_path=os.path.join(run_dir, f"{prefix}_checkpoint_test_acc.png"),
            show=False,
        )

        plot_curve(
            [row["train_acc"] for row in checkpoint_rows],
            title=f"{prefix} train accuracy every {checkpoint_every} epochs",
            xlabel="Checkpoint index",
            ylabel="Accuracy",
            label="train accuracy",
            save_path=os.path.join(run_dir, f"{prefix}_checkpoint_train_acc.png"),
            show=False,
        )

    return train_losses, train_sparsities, checkpoint_rows


# =========================================================
# one unified experiment
# =========================================================
def run_one_pruning_experiment(
    dataset_name,
    seed,
    base_dir,
    model_type,
    method_name,
    final_spar,
    method_kwargs=None,
    post_prune_train_epochs=100,
    checkpoint_every=5,
    batch_size=128,
    download=True,
    save_model=True,
    no_progress=False,
):
    if method_kwargs is None:
        method_kwargs = {}

    set_seed(seed)

    run_dir = make_pruning_run_dir(
        base_dir=base_dir,
        dataset_name=dataset_name,
        model_type=model_type,
        method_name=method_name,
        seed=seed,
        final_spar=final_spar,
    )

    info, task, n_classes, train_loader, train_loader_at_eval, test_loader = getTrainingDataLoaders(
        dataset_name,
        download=download,
        BATCH_SIZE=batch_size,
    )

    config = {
        "dataset_name": dataset_name,
        "seed": seed,
        "model_type": model_type,
        "method_name": method_name,
        "final_spar": final_spar,
        "method_kwargs": method_kwargs,
        "post_prune_train_epochs": post_prune_train_epochs,
        "checkpoint_every": checkpoint_every,
        "batch_size": batch_size,
    }
    save_json(config, os.path.join(run_dir, "config.json"))

    # -------------------------
    # dense base model
    # -------------------------
    set_seed(seed)
    dense_model = build_model(model_type, n_classes).to(device)


    # -------------------------
    # method-specific pruning stage
    # -------------------------
    set_seed(seed)
    method_name_lower = method_name.lower()

    if method_name_lower in {"wtl", "popup", "poppedup", "edge_popup"}:
        pruned_model, prune_losses, prune_sparsities = run_popup_pruning_phase(
            dense_model=dense_model,
            final_spar=final_spar,
            train_loader=train_loader,
            task=task,
            n_classes=n_classes,
            run_dir=run_dir,
            method_kwargs=method_kwargs,
            no_progress=no_progress,
        )
        sparsity_info = get_pruned_model_sparsity_info(pruned_model)
        save_json(sparsity_info, os.path.join(run_dir, "mask_sparsity.json"))

    else:
        keep_masks, imp_net = compute_keep_masks(
            method_name=method_name,
            final_spar=final_spar,
            model=dense_model,
            train_loader=train_loader,
            device=device,
            n_classes=n_classes,
            task=task,
            method_kwargs=method_kwargs,
        )

        if method_name_lower == "imp" and imp_net is not None:
            pruned_model = imp_net.to(device)
        else:
            ordered_masks = ordered_masks_from_keep_masks(dense_model, keep_masks)
            pruned_model = MaskedNetwork(dense_model, masks=ordered_masks).to(device)

        sparsity_info = get_pruned_model_sparsity_info(pruned_model)
        save_json(sparsity_info, os.path.join(run_dir, "mask_sparsity.json"))

    # -------------------------
    # eval immediately after pruning
    # -------------------------
    post_prune_pretrain_metrics = evaluate_model(
        model=pruned_model,
        train_loader_at_eval=train_loader_at_eval,
        test_loader=test_loader,
        task=task,
        n_classes=n_classes,
        data_flag=dataset_name,
    )
    save_json(
        post_prune_pretrain_metrics,
        os.path.join(run_dir, "post_prune_pretrain_metrics.json"),
    )

    # -------------------------
    # post-pruning training
    # -------------------------
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, pruned_model.parameters())
    )

    train_losses, train_sparsities, checkpoint_rows = train_pruned_model_with_checkpoints(
        model=pruned_model,
        total_epochs=post_prune_train_epochs,
        train_loader=train_loader,
        train_loader_at_eval=train_loader_at_eval,
        test_loader=test_loader,
        optimizer=optimizer,
        task=task,
        n_classes=n_classes,
        data_flag=dataset_name,
        checkpoint_every=checkpoint_every,
        run_dir=run_dir,
        prefix="post_prune",
        no_progress=no_progress,
        return_losses=True,
        return_sparsity=True,
    )

    # -------------------------
    # final metrics
    # -------------------------
    final_metrics = evaluate_model(
        model=pruned_model,
        train_loader_at_eval=train_loader_at_eval,
        test_loader=test_loader,
        task=task,
        n_classes=n_classes,
        data_flag=dataset_name,
    )
    save_json(final_metrics, os.path.join(run_dir, "final_metrics.json"))

    if save_model:
        torch.save(pruned_model.state_dict(), os.path.join(run_dir, "model_state_dict.pt"))

    result = {
        "dataset_name": dataset_name,
        "seed": seed,
        "model_type": model_type,
        "method_name": method_name,
        "final_spar": final_spar,
        "run_dir": run_dir,

        "sparsity": float(sparsity_info["sparsity"]),
        "zero_count": int(sparsity_info["zero_count"]),
        "total_count": int(sparsity_info["total_count"]),

        "pretrain_train_loss": float(post_prune_pretrain_metrics["train_loss"]),
        "pretrain_train_acc": float(post_prune_pretrain_metrics["train_acc"]),
        "pretrain_test_loss": float(post_prune_pretrain_metrics["test_loss"]),
        "pretrain_test_acc": float(post_prune_pretrain_metrics["test_acc"]),

        "final_train_loss": float(final_metrics["train_loss"]),
        "final_train_acc": float(final_metrics["train_acc"]),
        "final_test_loss": float(final_metrics["test_loss"]),
        "final_test_acc": float(final_metrics["test_acc"]),
    }

    for row in checkpoint_rows:
        ep = row["epoch"]
        result[f"train_loss_epoch_{ep}"] = float(row["train_loss"])
        result[f"train_acc_epoch_{ep}"] = float(row["train_acc"])
        result[f"test_loss_epoch_{ep}"] = float(row["test_loss"])
        result[f"test_acc_epoch_{ep}"] = float(row["test_acc"])

    return result


# =========================================================
# run many experiments and build one comparison table
# =========================================================
def run_pruning_comparison_suite(
    dataset_name,
    seeds,
    sparsities,
    base_dir,
    model_types,
    methods_config,
    post_prune_train_epochs=100,
    checkpoint_every=5,
    batch_size=64,
    download=True,
    no_progress=False,
):
    all_results = []

    for model_type in model_types:
        for final_spar in sparsities:
            for seed in seeds:
                for mcfg in methods_config:
                    method_name = mcfg["method_name"]
                    method_kwargs = mcfg.get("method_kwargs", {})

                    print(
                        f"\n=== RUNNING {model_type} | {method_name} | "
                        f"sparsity={final_spar} | seed={seed} ==="
                    )

                    result = run_one_pruning_experiment(
                        dataset_name=dataset_name,
                        seed=seed,
                        base_dir=base_dir,
                        model_type=model_type,
                        method_name=method_name,
                        final_spar=final_spar,
                        method_kwargs=method_kwargs,
                        post_prune_train_epochs=post_prune_train_epochs,
                        checkpoint_every=checkpoint_every,
                        batch_size=batch_size,
                        download=download,
                        save_model=True,
                        no_progress=no_progress,
                    )
                    all_results.append(result)

                    df = pd.DataFrame(all_results)
                    ensure_dir(base_dir)
                    df.to_csv(os.path.join(base_dir, "all_pruning_results.csv"), index=False)

    df = pd.DataFrame(all_results)
    df.to_csv(os.path.join(base_dir, "all_pruning_results.csv"), index=False)

    group_cols = ["dataset_name", "model_type", "method_name", "final_spar"]
    summary = (
        df.groupby(group_cols)
          .agg(
              final_test_acc_mean=("final_test_acc", "mean"),
              final_test_acc_std=("final_test_acc", "std"),
              sparsity_mean=("sparsity", "mean"),
              pretrain_test_acc_mean=("pretrain_test_acc", "mean"),
          )
          .reset_index()
    )

    summary.to_csv(os.path.join(base_dir, "summary_by_method.csv"), index=False)

    return df, summary

plotting helpers

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def run_label_from_row(row, include_seed=False, include_sparsity=False, include_model=False):
    parts = [row["method_name"]]

    if include_model:
        parts.append(str(row["model_type"]))
    if include_sparsity:
        parts.append(f"s={row['final_spar']}")
    if include_seed:
        parts.append(f"seed={row['seed']}")

    return " | ".join(parts)

def plot_post_prune_losses(smoke_df, out_dir, filename="post_prune_losses_multiline.png"):
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(9, 6))

    plotted_any = False

    for _, row in smoke_df.iterrows():
        run_dir = row["run_dir"]
        csv_path = os.path.join(run_dir, "post_prune_train_losses.csv")

        if not os.path.exists(csv_path):
            print("Missing:", csv_path)
            continue

        loss_df = pd.read_csv(csv_path)

        x = loss_df["step"].tolist()
        y = loss_df["loss"].tolist()

        label = run_label_from_row(row)
        plt.plot(x, y, marker="o", label=label)
        plotted_any = True

    plt.title("Post-Prune Training Loss")
    plt.xlabel("Training Step")
    plt.ylabel("Loss")

    if plotted_any:
        plt.legend()

    out_path = os.path.join(out_dir, filename)
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()

    print("Saved:", out_path)
    return out_path

def plot_post_prune_checkpoint_test_acc(smoke_df, out_dir, filename="post_prune_test_acc_multiline.png"):
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(9, 6))

    plotted_any = False

    for _, row in smoke_df.iterrows():
        run_dir = row["run_dir"]
        csv_path = os.path.join(run_dir, "post_prune_checkpoint_metrics.csv")

        if not os.path.exists(csv_path):
            print("Missing:", csv_path)
            continue

        ckpt_df = pd.read_csv(csv_path)

        x = ckpt_df["epoch"].tolist()
        y = ckpt_df["test_acc"].tolist()

        label = run_label_from_row(row)
        plt.plot(x, y, marker="o", label=label)
        plotted_any = True

    plt.title("Post-Prune Checkpoint Test Accuracy")
    plt.xlabel("Post-Prune Training Epoch")
    plt.ylabel("Test Accuracy")

    # Set x-axis ticks to integers only
    plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))

    if plotted_any:
        plt.legend()

    out_path = os.path.join(out_dir, filename)
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()

    print("Saved:", out_path)
    return out_path

def plot_post_prune_pretrain_test_acc_bar(smoke_df, out_dir, filename="post_prune_pretrain_test_acc_bar.png"):
    os.makedirs(out_dir, exist_ok=True)

    labels = []
    values = []

    for _, row in smoke_df.iterrows():
        run_dir = row["run_dir"]
        json_path = os.path.join(run_dir, "post_prune_pretrain_metrics.json")

        if not os.path.exists(json_path):
            print("Missing:", json_path)
            continue

        with open(json_path, "r") as f:
            metrics = json.load(f)

        labels.append(run_label_from_row(row))
        values.append(metrics["test_acc"])

    plt.figure(figsize=(8, 5))
    plt.bar(labels, values)
    plt.title("Test Accuracy Immediately After Pruning")
    plt.xlabel("Method")
    plt.ylabel("Test Accuracy")

    out_path = os.path.join(out_dir, filename)
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()

    print("Saved:", out_path)
    return out_path

def make_smoke_test_plots(smoke_df, base_dir):
    plots_dir = os.path.join(base_dir, "smoke_plots")
    os.makedirs(plots_dir, exist_ok=True)

    out1 = plot_post_prune_losses(
        smoke_df,
        out_dir=plots_dir,
        filename="1_post_prune_losses_multiline.png"
    )

    out2 = plot_post_prune_checkpoint_test_acc(
        smoke_df,
        out_dir=plots_dir,
        filename="2_post_prune_test_acc_multiline.png"
    )

    out3 = plot_post_prune_pretrain_test_acc_bar(
        smoke_df,
        out_dir=plots_dir,
        filename="3_post_prune_pretrain_test_acc_bar.png"
    )

    return {
        "loss_plot": out1,
        "checkpoint_test_acc_plot": out2,
        "pretrain_test_acc_bar": out3,
    }

test--------

In [ ]:
SMOKE_BASE_DIR = "/content/drive/MyDrive/experiments/pruning_compare_smoke"
SMOKE_DATASET = "cifar10"

SMOKE_SEEDS = [0]
SMOKE_SPARSITIES = [0.9]
SMOKE_MODEL_TYPES = ["mlp"]   # use mlp first because it's faster/simpler

SMOKE_POST_PRUNE_TRAIN_EPOCHS = 2
SMOKE_CHECKPOINT_EVERY = 1
SMOKE_BATCH_SIZE = 64

smoke_methods_config = [
    {
        "method_name": "wtl",
        "method_kwargs": {
            "prune_epochs": 2,

        },
    },
    {
        "method_name": "snip",
        "method_kwargs": {
            "samples_per_class": 5,
            "num_iters": 1,
        },
    },
    {
        "method_name": "grasp",
        "method_kwargs": {
            "samples_per_class": 5,
            "num_iters": 1,
            "T": 200,
            "reinit": False,
        },
    },
    {
        "method_name": "imp",
        "method_kwargs": {
            "tau": 10,
            "L_max": 2,         # very small number of prune rounds
            "iter_epochs": 1,   # just for smoke testing, not paper-faithful
            "rewind_to_init": False,
            "prune_global": False,
        },
    },
]

smoke_df, smoke_summary = run_pruning_comparison_suite(
    dataset_name=SMOKE_DATASET,
    seeds=SMOKE_SEEDS,
    sparsities=SMOKE_SPARSITIES,
    base_dir=SMOKE_BASE_DIR,
    model_types=SMOKE_MODEL_TYPES,
    methods_config=smoke_methods_config,
    post_prune_train_epochs=SMOKE_POST_PRUNE_TRAIN_EPOCHS,
    checkpoint_every=SMOKE_CHECKPOINT_EVERY,
    batch_size=SMOKE_BATCH_SIZE,
    download=True,
    no_progress=False,
)

plot_paths = make_smoke_test_plots(smoke_df, SMOKE_BASE_DIR)
print(plot_paths)

print("Smoke test finished.")
print(smoke_df.head())
print(smoke_summary)

import os

expected_top_level = [
    os.path.join(SMOKE_BASE_DIR, "all_pruning_results.csv"),
    os.path.join(SMOKE_BASE_DIR, "summary_by_method.csv"),
]

for path in expected_top_level:
    print(path, "->", os.path.exists(path))

for _, row in smoke_df.iterrows():
    run_dir = row["run_dir"]
    print("\nRUN DIR:", run_dir)
    for fname in [
        "config.json",
        "dense_init_metrics.json",
        "mask_sparsity.json",
        "post_prune_pretrain_metrics.json",
        "final_metrics.json",
        "post_prune_checkpoint_metrics.csv",
    ]:
        print(f"  {fname}: {os.path.exists(os.path.join(run_dir, fname))}")


=== RUNNING mlp | wtl | sparsity=0.9 | seed=0 ===
0.0% DONE


 51%|█████     | 398/782 [00:03<00:03, 106.91it/s]


KeyboardInterrupt: 

In [ ]:
plot_paths = make_smoke_test_plots(smoke_df, SMOKE_BASE_DIR)

Saved: /content/drive/MyDrive/experiments/pruning_compare_smoke/smoke_plots/1_post_prune_losses_multiline.png
Saved: /content/drive/MyDrive/experiments/pruning_compare_smoke/smoke_plots/2_post_prune_test_acc_multiline.png
Saved: /content/drive/MyDrive/experiments/pruning_compare_smoke/smoke_plots/3_post_prune_pretrain_test_acc_bar.png


In [ ]:
BASE_DIR = "/content/drive/MyDrive/experiments/pruning_compare_cifar"
DATASET_NAME = "cifar10"

SEEDS = [0, 1, 25]
MODEL_TYPES = ["vgg"]
SPARSITIES = [0.95, 0.90, 0.85, 0.80]

POST_PRUNE_TRAIN_EPOCHS = 60
CHECKPOINT_EVERY = 5
BATCH_SIZE = 64

methods_config = [
    {
        "method_name": "wtl",
        "method_kwargs": {
            "prune_epochs": 60,

        },
    },
    {
        "method_name": "snip",
        "method_kwargs": {
            "samples_per_class": 25,
            "num_iters": 1,
        },
    },
    {
        "method_name": "grasp",
        "method_kwargs": {
            "samples_per_class": 25,
            "num_iters": 1,
            "T": 200,
            "reinit": False,
        },
    },
    {
        "method_name": "imp",
        "method_kwargs": {
            "tau": 100,#pretraining steps
            "L_max": 3,
            "iter_epochs": 20,
            "rewind_to_init": False,
            "prune_global": False,
        },
    },
]

df, summary = run_pruning_comparison_suite(
    dataset_name=DATASET_NAME,
    seeds=SEEDS,
    sparsities=SPARSITIES,
    base_dir=BASE_DIR,
    model_types=MODEL_TYPES,
    methods_config=methods_config,
    post_prune_train_epochs=POST_PRUNE_TRAIN_EPOCHS,
    checkpoint_every=CHECKPOINT_EVERY,
    batch_size=BATCH_SIZE,
    download=True,
    no_progress=False,
)

plot_paths = make_smoke_test_plots(df, SMOKE_BASE_DIR)
print(plot_paths)

print("Saved master results to:")
print(f"{BASE_DIR}/all_pruning_results.csv")
print(f"{BASE_DIR}/summary_by_method.csv")

plot_paths = make_smoke_test_plots(df, BASE_DIR)
print(plot_paths)

summary


=== RUNNING vgg | wtl | sparsity=0.95 | seed=0 ===
0.0% DONE


100%|██████████| 782/782 [00:16<00:00, 46.45it/s]


1.7% DONE


100%|██████████| 782/782 [00:15<00:00, 49.74it/s]


3.3% DONE


...

100%|██████████| 782/782 [00:08<00:00, 87.60it/s]


80.0% DONE


100%|██████████| 782/782 [00:08<00:00, 87.20it/s]


train loss: 0.0313 acc: 0.9897
test loss: 0.4194 acc: 0.9109
train loss: 0.0308 acc: 0.9892
test loss: 0.4194 acc: 0.9109
Saved: /content/drive/MyDrive/experiments/pruning_compare_smoke/smoke_plots/1_post_prune_losses_multiline.png
Saved: /content/drive/MyDrive/experiments/pruning_compare_smoke/smoke_plots/2_post_prune_test_acc_multiline.png
Saved: /content/drive/MyDrive/experiments/pruning_compare_smoke/smoke_plots/3_post_prune_pretrain_test_acc_bar.png
{'loss_plot': '/content/drive/MyDrive/experiments/pruning_compare_smoke/smoke_plots/1_post_prune_losses_multiline.png', 'checkpoint_test_acc_plot': '/content/drive/MyDrive/experiments/pruning_compare_smoke/smoke_plots/2_post_prune_test_acc_multiline.png', 'pretrain_test_acc_bar': '/content/drive/MyDrive/experiments/pruning_compare_smoke/smoke_plots/3_post_prune_pretrain_test_acc_bar.png'}
Saved master results to:
/content/drive/MyDrive/experiments/pruning_compare_cifar/all_pruning_results.csv
/content/drive/MyDrive/experiments/pruning_

,dataset_name,model_type,method_name,final_spar,final_test_acc_mean,final_test_acc_std,sparsity_mean,pretrain_test_acc_mean
0,cifar10,vgg,grasp,0.80,0.897800,0.004161,0.800000,0.100000
1,cifar10,vgg,grasp,0.85,0.895433,0.001266,0.850000,0.100000
2,cifar10,vgg,grasp,0.90,0.899133,0.000808,0.900000,0.100000
3,cifar10,vgg,grasp,0.95,0.893333,0.003650,0.950000,0.100000
4,cifar10,vgg,imp,0.80,0.910433,0.000451,0.800001,0.100000
5,cifar10,vgg,imp,0.85,0.908233,0.003233,0.850001,0.100000
6,cifar10,vgg,imp,0.90,0.904067,0.002994,0.900001,0.100000
7,cifar10,vgg,imp,0.95,0.890767,0.002316,0.950001,0.100000
8,cifar10,vgg,snip,0.80,0.905100,0.004681,0.800000,0.099767
9,cifar10,vgg,snip,0.85,0.907367,0.001662,0.850000,0.098067
